# الدرس 07 - نمط تصميم التخطيط

تعرض هذه الدفتر نموذج **نمط تصميم التخطيط** لوكلاء الذكاء الاصطناعي باستخدام إطار عمل Microsoft Agent.  
ستتعلم كيفية تقسيم طلبات السفر المعقدة إلى مهام فرعية منظمة، وتعيينها إلى وكلاء متخصصين،  
وتنفيذ الخطة الناتجة — كل ذلك باستخدام المخرجات المنظمة المدعومة بنماذج Pydantic.


## الإعداد


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## تقسيم المهمة

يعد تقسيم المهام هو جوهر نمط تصميم التخطيط. بدلاً من طلب من وكيل واحد التعامل مع طلب معقد من البداية إلى النهاية، نقوم بتقسيم المشكلة إلى **مهام فرعية** أصغر ومحددة جيدًا. يتم تعيين كل مهمة فرعية لوكيل متخصص (مثل الرحلات، الفنادق، الأنشطة) مع وجود أولوية واضحة وترتيب اعتمادية.

يوفر هذا النهج العديد من الفوائد:
- **الوضوح**: كل مهمة فرعية لها مسؤولية واحدة.
- **التوازي**: يمكن تشغيل المهام الفرعية المستقلة بشكل متزامن.
- **الموثوقية**: يتم عزل الإخفاقات في المهام الفرعية الفردية.
- **تتبع الميزانية**: يتم تقدير التكاليف لكل مهمة فرعية وتُجمع معًا.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## إنشاء وكيل تخطيط مع إخراج منظم

يعمل وكيل التخطيط كـ **منسق استقبال**. بناءً على طلب سفر عالي المستوى، يقوم بإنتاج `TravelPlan` منظم — يقوم بتفكيك الطلب إلى مهام فرعية، وتحديد الأولويات، وتحديد التبعيات بحيث يمكن للكونسيرج أو طبقة التنفيذ القيام بالعمل.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## تنفيذ خطة باستخدام أدوات متخصصة

بمجرد أن ينتج وكيل الاستقبال خطة منظمة، يقوم **وكيل الكونسيرج** بتنفيذها. تتعامل كل أداة متخصصة مع فئة واحدة من المهام الفرعية (الرحلات الجوية، الفنادق، الأنشطة). يقوم الكونسيرج بالتكرار عبر المهام الفرعية للخطة وفقًا لترتيب التبعية ويرسل كل واحدة إلى الأداة المناسبة.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## الملخص

في هذا الدرس تعلمت **نموذج تصميم التخطيط** لوكلاء الذكاء الاصطناعي:

1. **تفكيك المهمة** — يقوم وكيل التخطيط في مكتب الاستقبال بتقسيم طلب السفر المعقد إلى مهام فرعية منظمة باستخدام نماذج Pydantic، مخصصًا كل مهمة إلى وكيل متخصص مع تحديد الأولويات والاعتمادات.
2. **الناتج المنظم** — من خلال تمرير `response_format` يُرجع الوكيل كائن `TravelPlan` مُحقق بدلاً من نص حر، مما يجعل معالجة البيانات اللاحقة موثوقة.
3. **تنفيذ الخطة** — يقوم وكيل الكونسييرج بالتكرار عبر المهام الفرعية باستخدام أدوات متخصص (`book_flight`، `reserve_hotel`، `book_activity`) لتنفيذ الخطة وتقرير النتائج.

يفصل هذا النموذج بين *ما يجب القيام به* (التخطيط) و*كيفية القيام به* (التنفيذ)، مما يجعل الوكلاء أكثر تركيبية، وقابلين للاختبار، وأسهل في التوسعة.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنبيه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة الآلية [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى لتحقيق الدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الموثوق به. بالنسبة للمعلومات الهامة، يُنصح بالاستعانة بالترجمة المهنية البشرية. نحن غير مسؤولين عن أي سوء فهم أو تفسيرات خاطئة تنشأ من استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
